# Part 1.7 Interrupts

After this notebook you can pause a graph for a human decision with `interrupt()` and
resume it with `Command(resume=...)`.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

A crushed bag with an $800 declared value pays more than $250, so policy sends it to a
person. The finished graph pauses at `human_review`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from kea.claims.graph import builder

store = InMemoryStore()
graph = builder.compile(checkpointer=InMemorySaver(), store=store)


def claim(photo: str, passenger_id: str, flight_number: str, declared_value_usd: float) -> dict:
    return {
        "photo": photo,  # a file name under data/bag-photos, the same input Studio takes
        "passenger_id": passenger_id,
        "flight_number": flight_number,
        "declared_value_usd": declared_value_usd,
    }

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "claim-1002"}}
paused = graph.invoke(claim("crushed-01.jpg", "pax-1002", "KA202", 800), config)

print("next node:", graph.get_state(config).next)
paused["__interrupt__"][0].value

In [ ]:
approval = {"approved": True, "payout_usd": 300, "note": "Approved at $300 after review."}
resumed = graph.invoke(Command(resume=approval), config)

print(resumed["status"], resumed["payout_usd"])
print(resumed["letter"])

<img src="../../assets/screenshots/p1-07-studio-interrupt.png" alt="Studio paused on humanreview: the interrupt payload (claim, proposed payout, reasons) and the resume box." width="900">

## Build it

The review step is one node. It calls `interrupt()` with what the reviewer needs to
see, and the value passed to `Command(resume=...)` becomes what `interrupt()` returns.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import interrupt

from kea.claims.models import PolicyDecision, ReviewDecision
from kea.claims.policy import settle


class State(TypedDict, total=False):
    policy: dict
    payout_usd: float


def human_review(state: State) -> dict:
    policy = PolicyDecision.model_validate(state["policy"])
    answer = interrupt({"proposed_payout_usd": policy.payout_usd, "reasons": policy.review_reasons})
    review = ReviewDecision.model_validate(answer)
    return {"payout_usd": settle(policy, review)}


builder = StateGraph(State)
builder.add_node("human_review", human_review)
builder.add_edge(START, "human_review")
builder.add_edge("human_review", END)
review_graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
policy = PolicyDecision(payout_usd=350, review_reasons=["payout $350 is above $250"])
config = {"configurable": {"thread_id": "review-1"}}

paused = review_graph.invoke({"policy": policy.model_dump()}, config)
print(paused["__interrupt__"][0].value)

done = review_graph.invoke(Command(resume={"approved": True, "payout_usd": 300}), config)
print("payout:", done["payout_usd"])

## Under the hood

<img src="../../assets/slides/langgraph-human-in-the-loop.jpg" width="720">

- `interrupt(payload)` saves a checkpoint and stops the run. The payload comes back to
  the caller under `__interrupt__`.
- **On resume the node re-runs from the top.** So nothing with side effects belongs
  before `interrupt()`. In the claims graph, the Store write happens later, in
  `record_claim`.
- The resume value is validated as a `ReviewDecision`, so a malformed answer is rejected
  before it reaches the letter. A denial cannot carry a payout:

In [ ]:
from pydantic import ValidationError

try:
    ReviewDecision(approved=False, payout_usd=50)
except ValidationError as error:
    print(error.errors()[0]["msg"])

- Interrupts need a checkpointer. No saved state, no place to wait.

Docs: [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts).

## Recap

- `interrupt()` pauses a graph; `Command(resume=...)` continues it.
- The node restarts from its top on resume, so keep side effects out of it.
- Validate the human's answer like any other input.

**Next:** [Part 1.8 Time travel](08_time_travel.ipynb)